# Imports

In [1]:
import numpy as np
from scipy.spatial.transform import Rotation as rotate

from functions.representation import Object, RectangularPrism
from functions.vectors import cosine_similarity, greatest_landmark_distance, find_center_point_LWLC, find_axis_of_rotation, same_object
from functions.plotting import  add_point, add_landmarks, add_axis, add_object, add_frame, prepare_rotation_graphs

import plotly.graph_objects as go
import plotly.offline as pyo
from plotly.subplots import make_subplots
import copy

# Initialize Plotly for offline mode in Jupyter Notebook
pyo.init_notebook_mode(connected=True)

# Model Variables

In [2]:
total_run_time = 0

# timing
production_time = 50                        # 50ms
propositional_difficulty_time = 1           # 1-3 ms
object_encoding_time = 300                  # 150-300ms for each object

# rotation
max_step_size = 30
min_step_size = 2
step_size_decrease = 1/2                    # rate at which step size decreases
decrease_begins = 1/2                       # step size begins to decrease after [decrease_begins] of the total rotation is complete
# x, y, z errors

# decision
repeat_threshold = 2         # 0-4 repeated steps

# similarity thresholds
landmark_angle_threshold = 0.975
landmark_distance_threshold = 0.3
object_angle_threshold = 0.975
object_distance_threshold = landmark_distance_threshold * 2

# 1. Representation

This is the case I'll model:

![image](test.jpg)

*150 degree diff in pic

In [ ]:
r = rotate.from_rotvec([0, 0, np.deg2rad(150)])                               # 150 deg rotation around z-axis
# r = rotate.from_rotvec([np.deg2rad(150), np.deg2rad(60), np.deg2rad(90)])       # crazy rotation :o

# x: right is positive, y: further away is positive, z: up is positive

# create original geons
g1 = RectangularPrism(2, np.array([-1, -1, 0]))             # 45 deg angle front left, no z info
g2 = RectangularPrism(3, np.array([0, 0, -1]))              # down
g3 = RectangularPrism(2, np.array([1, 1, 0]))               # 45 deg angle back right, no z info
g4 = RectangularPrism(1, np.array([1, -1, 0]))              # 45 deg angle front right, no z info

# create original object and relations
original_object = Object(
    geons = [g1,g2,g3,g4],
    landmark_geon_index = 0                    # i.e. landmark is g1
)

# create target geons (same as original, but with rotation applied)
g1 = RectangularPrism(2, r.apply(np.array([-1, -1, 0])))
g2 = RectangularPrism(3, r.apply(np.array([0, 0, -1])))
g3 = RectangularPrism(2, r.apply(np.array([1, 1, 0])))
g4 = RectangularPrism(1, r.apply(np.array([1, -1, 0])))

# # MIRRORED target object
# g1 = RectangularPrism(2, r.apply(np.array([-1, -1, 0])))
# g2 = RectangularPrism(3, r.apply(np.array([0, 0, -1])))
# g3 = RectangularPrism(2, r.apply(np.array([1, 1, 0])))
# g4 = RectangularPrism(1, r.apply(np.array([-1, 1, 0])))

target_object = Object(
    geons = [g1,g2,g3,g4],
    landmark_geon_index = 0                    # i.e. landmark is g1
)

# make copy of original object
original_object_reset = copy.deepcopy(original_object)

# calculate axis of rotation, direction of rotation, and total angular disparity
center_point = np.array([0,0,0])
axis_of_rotation, direction, angle = find_axis_of_rotation(original_object, target_object, center_coords=center_point)
total_angular_disparity = angle

# add time needed for representation
total_run_time = total_run_time + (2 * object_encoding_time)

# 2. Landmarking

In [4]:
# add time needed for landmarking
total_run_time = total_run_time + production_time                                                                 # check geon
total_run_time = total_run_time + production_time + (total_angular_disparity * propositional_difficulty_time)     # check spatial connection

# 3. Rotation & 4. Decision (Loop)

In [ ]:
repeat_count = -1
same = False
while not same and repeat_count < repeat_threshold:

    # count repeated rotation step
    if not same:
        repeat_count += 1

        if repeat_count > 0:
            print("REPEAT #" + str(repeat_count))

    # prepare graphs
    axis_animation_fig, overlap_animation_fig, sidebyside_animation_fig = prepare_rotation_graphs(original_object, target_object, axis_of_rotation, center_point, production_time)
    axis_animation = []
    overlap_animation = []
    sidebyside_animation = []

    # set landmark vectors, angular disparity
    original_landmark_geon_vector = original_object.get_landmark_geon().get_vector()
    target_landmark_geon_vector = target_object.get_landmark_geon().get_vector()
    curr_step_angular_disparity = total_angular_disparity

    loop_count = 0
    while cosine_similarity(original_landmark_geon_vector, target_landmark_geon_vector) < landmark_angle_threshold or greatest_landmark_distance(original_object.get_landmark_endpoints(), target_object.get_landmark_endpoints()) > landmark_distance_threshold:     # checking cosine similarity and distance between landmarks

        # find best axis/direction of rotation, angular disparity
        axis_of_rotation, direction, curr_step_angular_disparity = find_axis_of_rotation(original_object, target_object, center_coords=center_point, prev_axis=axis_of_rotation, prev_direction=direction, prev_angle=np.deg2rad(curr_step_angular_disparity), total_angular_disparity=total_angular_disparity)
        curr_step_angular_disparity = np.rad2deg(curr_step_angular_disparity)

        # calculate step size
        # step_size = max_step_size if (curr_step_angular_disparity > total_angular_disparity * decrease_begins) else (curr_step_angular_disparity * step_size_decrease)
        step_size = curr_step_angular_disparity * step_size_decrease
        step_size = min(max(step_size, min_step_size), max_step_size)
        # step_size = np.rad2deg(prev_angle) * step_size_decrease
        # print(step_size)

        r = rotate.from_rotvec(direction * np.deg2rad(step_size) * axis_of_rotation)        # quaternion representing step size rotation around calculated axis

        # apply rotation to original object
        original_object.rotate(r)

        # update original_landmark_vector
        original_landmark_geon_vector = original_object.get_landmark_geon().get_vector()

        # add animation frame to graph
        add_frame(axis_animation, original_object, object_name="Original Object", landmark_name="Original Landmarks", axis_of_rotation=axis_of_rotation, object_colour='blue', landmark_colour='purple', axis_colour='green', axis_scale=2)

        original_centerpoint_vec = find_center_point_LWLC(original_object)
        copy_og_obj = copy.deepcopy(original_object)
        copy_og_obj.update_start_coords(copy_og_obj.start_coords - original_centerpoint_vec)
        add_frame(overlap_animation, copy_og_obj, object_name="Original Object", object_colour='blue', axis_scale=2)
        add_frame(sidebyside_animation, copy_og_obj, object_name="Original Object", object_colour='blue', axis_scale=2)

        loop_count += 1
        total_run_time += production_time * 3

        if loop_count > 100:
            break

        # print(cosine_similarity(original_landmark_geon_vector, target_landmark_geon_vector))
        # print(greatest_landmark_distance(original_object.get_landmark_endpoints(), target_object.get_landmark_endpoints()))

    axis_animation_fig.frames = axis_animation
    axis_animation_fig.show()
    overlap_animation_fig.frames = overlap_animation
    overlap_animation_fig.show()
    sidebyside_animation_fig.frames = sidebyside_animation
    sidebyside_animation_fig.show()

    # check overall similarity

    same, similarity_check_run_time = same_object(original_object, target_object, object_angle_threshold, total_angular_disparity, production_time, propositional_difficulty_time)
    total_run_time += similarity_check_run_time

    if not same:

        # reset original object
        original_object = original_object_reset
        axis_of_rotation, direction, angle = find_axis_of_rotation(original_object, target_object, center_coords=center_point)
        total_angular_disparity = angle

print("DECISION:")
if same:
    print("same")
else:
    print("different")

print("\nRUN TIME:\n" + str(round(total_run_time/1000, 3)) + " seconds")

DECISION:
same

RUN TIME:
3.019 seconds
